# CardiaSense — LLM integration & forward architecture

**Educational technical walkthrough · notebook 5 of 5**

**Objectives:** provider/synthesizer narrative, personalization, chat, and two research directions

**You will need:** the `cardiasense` venv (run `./setup.sh`), the dataset at
`$CARDIASENSE_DATA_DIR` (downloaded by `setup.sh`), and a trained head for the
notebooks that use checkpoints (or run the smoke train in notebook 03 first).

> Educational / research prototype — **not a medical device**.

---




### Step 1 — From probabilities to a personalized narrative

The heads output probabilities — a patient wants *words*. The LLM integration
(`webapp/app/llm.py`) has two interchangeable providers that return the **same
JSON shape**:

- `LLMClient` — calls an OpenAI-compatible endpoint (`CARDIASENSE_LLM_*`), asked
  for strict JSON, grounded in the findings + patient context.
- `LLMSynthesizer` — an **offline rule-based fallback** built from a per-class
  knowledge base (medical / diet / environment outlines + context hints), so the
  product works with no API key at all.

Below we synthesize from a plausible finding vector and a short patient note.
Notice the **personalization** (context hints folded into the medical
recommendations) and the guardrail: every output carries the disclaimer.


In [1]:

import os, sys
from pathlib import Path

ROOT = Path("/home/tnzr/CardiaSense-AI")
sys.path.insert(0, str(ROOT / "ml"))
sys.path.insert(0, str(ROOT))

from cardia.config import DEFAULT_DATA_DIR   # loads .env if present
DATA = os.environ.get("CARDIASENSE_DATA_DIR") or DEFAULT_DATA_DIR
print("Dataset root:", DATA)


Dataset root: /home/tnzr/CardiaSense-AI/data/HLS-CMDS


In [2]:

from webapp.app import llm as llm_mod
import json

findings = [
    {"key": "heart_binary", "label": "Heart screening", "kind": "binary", "pred": "abnormal",
     "confidence": 0.82, "classes": ["normal", "abnormal"], "probs": [0.18, 0.82]},
    {"key": "lung_types", "label": "Lung typing", "kind": "multiclass", "pred": "Wheezing",
     "confidence": 0.71, "classes": ["Normal", "Wheezing", "Fine Crackles", "Coarse Crackles",
                                     "Rhonchi", "Pleural Rub"]},
]
out = llm_mod.generate_narrative(findings, patient_context="58y, asthma, mild palpitations")
print("provider:", out["provider"], "| model:", out["model"], "| personalized:", out["personalized"])
print("\nSUMMARY:", out["summary"])
print("\nFINDINGS:"); [print(" -", f) for f in out["findings"]]
for area, recs in out["recommendations"].items():
    print(f"\n[{area}]"); [print("  -", r) for r in recs]


provider: synthesizer | model: rule-based-synthesizer-v1 | personalized: True

SUMMARY: The model flagged at least one possible abnormality. A clinician should interpret these findings with your full history and an examination.

FINDINGS:
 - Heart screening: abnormal (confidence 82%)
 - Lung typing: Wheezing (confidence 71%)

[medical]
  - Personalized from your stated context: Patient context mentions asthma — airway findings should be weighed accordingly.; Patient reports palpitations — correlate with rhythm findings and ECG.
  - Evaluate for asthma/COPD with a clinician; use reliever inhalers as prescribed

[diet]
  - Avoid known food triggers if asthma is food-sensitive
  - Stay hydrated to keep airways moist

[environment]
  - Reduce smoke, dust and strong fumes
  - Keep indoor humidity moderate



### Step 2 — Follow-up conversation (chat semantics)

The report page's chat sends `{findings, patient_context, history}` to
`POST /api/llm/report`; the LLM (or synthesizer) answers with the same shape, so
follow-ups inherit the auscultation context. With a key set, the live model
handles the dialogue; without one, the synthesizer keeps the flow working.


In [3]:

if os.environ.get("CARDIASENSE_LLM_API_KEY"):
    narr = llm_mod.generate_narrative(findings, "asthma",
                                      [{"role": "user", "content": "should I worry about the wheeze?"}])
    print("provider:", narr["provider"], "|", narr.get("summary", "")[:160])
else:
    print("No CARDIASENSE_LLM_API_KEY — using the offline synthesizer (shown above). "
          "Set it (or a local Ollama base URL) to exercise the live LLM path.")


No CARDIASENSE_LLM_API_KEY — using the offline synthesizer (shown above). Set it (or a local Ollama base URL) to exercise the live LLM path.



### Step 3 — Two research directions worth your time

1. **Token/embedding architecture (done in part, notebook 04).** The encoder is
   already a tokenizer: reuse its patch/CLS embeddings as the audio interface to
   a language model (frozen encoder + linear projection, or patch tokens as soft
   prompts). The served Transformer is ~0.34 GFLOPs / 2 MB int8 — cheap enough to
   run on-device and let the cloud LLM do the talking.

2. **Whole-breathing-cycle modeling.** A 1–3 s snippet can miss the pattern;
   proper auscultation wants the whole cycle (the app's macro pass already uses
   the full 15 s clip). Open paths: envelope-based **cycle segmentation** with
   per-cycle embeddings + cross-cycle attention, **long-context encoders** for
   minute-long exams, and **cycle-aware training** to reduce short-clip label
   noise. Full design + evaluation plan in `docs/LLM_Integration.md`.
